In [ ]:
import sys
import gc
import json
from pathlib import Path

import torch

# Robust repo-root import (works in local + Colab)
repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root / "Efficient_Architecture" / "src").exists():
    repo_root = repo_root / "Efficient_Architecture"
sys.path.append(str(repo_root))

from src.utils.models import Qwen3, Qwen35, LFM2, IBM_Granite1b, IBM_Granite, GPT2
from data.preprocessing import c4_dataset
from src.utils.metrics import (
    profiling_test_suite,
    profile_prefill_decode,
    non_embedding_params,
    count_params,
)

# Configuration
LANGUAGE = "en"
SPLIT = "train"
NUM_EXAMPLES = 1  # keep profiling lightweight

OUT_DIR = Path("profiling_metrics")
TRACE_DIR = Path("profiling_traces")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_DIR.mkdir(parents=True, exist_ok=True)

model_classes = {
    "Qwen3": Qwen3,
    "Qwen3.5": Qwen35,
    "LFM2": LFM2,
    "IBM-G1B": IBM_Granite1b,
    "IBM-G350M": IBM_Granite,
    "GPT2": GPT2,
}

for name, ModelClass in model_classes.items():
    print(f"\n=== Profiling {name} ===")

    wrapper = None
    model = None
    result = {"_model": name, "_status": "ok", "_points": {}}

    try:
        wrapper = ModelClass()
        model = wrapper.model

        # Parameter counts are cheap; helpful for correlating capacity vs performance.
        total_p = count_params(model)
        non_emb_p = non_embedding_params(model)
        result["_params"] = {
            "total": total_p,
            "non_embedding": non_emb_p,
            "embedding_plus_head": total_p - non_emb_p,
        }

        # Try to move to GPU for GPU profiling. If it OOMs, we continue on CPU.
        if torch.cuda.is_available():
            try:
                model = model.to("cuda")
            except Exception as e:
                print(f"Could not move {name} to CUDA; profiling on CPU. Error: {e}")
                model = wrapper.model  # keep whatever device it was on

        for (read_len, gen_len) in profiling_test_suite:
            key = f"({read_len},{gen_len})"
            print(f"  - {key}")

            try:
                dataset = c4_dataset(split=SPLIT, language=LANGUAGE, tokenizer=wrapper.tokenizer)
                dataloader = dataset.process(num_examples=NUM_EXAMPLES, sequence_length=read_len, batch_size=1)
                batch = next(iter(dataloader))

                traces_subdir = TRACE_DIR / name
                trace_prefix = f"r{read_len}_g{gen_len}"

                point_result = profile_prefill_decode(
                    model,
                    batch,
                    decode_steps=gen_len,
                    traces_dir=str(traces_subdir),
                    trace_prefix=trace_prefix,
                )
                result["_points"][key] = point_result

            except torch.cuda.OutOfMemoryError as e:
                result["_points"][key] = {"_status": "oom", "error": str(e)}
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except RuntimeError as e:
                msg = str(e).lower()
                if "out of memory" in msg:
                    result["_points"][key] = {"_status": "oom", "error": str(e)}
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                else:
                    result["_points"][key] = {"_status": "error", "error": str(e)}
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    except Exception as e:
        result["_status"] = "error"
        result["_error"] = str(e)
        print(f"Failed to profile {name}: {e}")

    # Save per-model profiling JSON regardless of success
    out_path = OUT_DIR / f"{name}_profiling.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"Saved {out_path}")

    # Free model memory before next model
    del model
    del wrapper
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("profiling_metrics")
PLOT_DIR = Path("profiling_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Load all profiling result files
result_files = sorted(OUT_DIR.glob("*_profiling.json"))
if not result_files:
    raise FileNotFoundError(f"No profiling JSON files found in {OUT_DIR}")

all_results = {}
for fp in result_files:
    with open(fp, "r") as f:
        obj = json.load(f)
    model_name = obj.get("_model", fp.stem.replace("_profiling", ""))
    all_results[model_name] = obj

# Use suite order for x-axis consistency
x_labels = [f"({r},{g})" for (r, g) in profiling_test_suite]

metrics_to_plot = [
    ("prefill_s", "Prefill Latency (s)", "prefill_latency.png"),
    ("decode_s", "Decode Latency (s)", "decode_latency.png"),
    ("prefill_peak_gpu_gb", "Prefill Peak GPU Memory (GB)", "prefill_gpu_peak.png"),
    ("decode_peak_gpu_gb", "Decode Peak GPU Memory (GB)", "decode_gpu_peak.png"),
    ("prefill_cpu_peak_mb", "Prefill Peak CPU Memory (MB)", "prefill_cpu_peak.png"),
    ("decode_cpu_peak_mb", "Decode Peak CPU Memory (MB)", "decode_cpu_peak.png"),
]

for key, ylabel, filename in metrics_to_plot:
    plt.figure(figsize=(10, 5))

    for model_name, obj in all_results.items():
        pts = obj.get("_points", {})
        ys = []
        for x in x_labels:
            item = pts.get(x, {})
            if isinstance(item, dict) and key in item:
                ys.append(item[key])
            else:
                ys.append(float("nan"))

        plt.plot(range(len(x_labels)), ys, marker="o", label=model_name)

    plt.xticks(range(len(x_labels)), x_labels, rotation=45)
    plt.xlabel("(read_len, gen_len)")
    plt.ylabel(ylabel)
    plt.title(ylabel)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = PLOT_DIR / filename
    plt.savefig(out_path)
    plt.show()
    print(f"Saved {out_path}")

# Combined 2x2 quick comparison view
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
combined = [
    ("prefill_s", "Prefill Latency (s)", axs[0, 0]),
    ("decode_s", "Decode Latency (s)", axs[0, 1]),
    ("prefill_peak_gpu_gb", "Prefill Peak GPU Mem (GB)", axs[1, 0]),
    ("decode_peak_gpu_gb", "Decode Peak GPU Mem (GB)", axs[1, 1]),
]

for key, title, ax in combined:
    for model_name, obj in all_results.items():
        pts = obj.get("_points", {})
        ys = []
        for x in x_labels:
            item = pts.get(x, {})
            if isinstance(item, dict) and key in item:
                ys.append(item[key])
            else:
                ys.append(float("nan"))
        ax.plot(range(len(x_labels)), ys, marker="o", label=model_name)

    ax.set_title(title)
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=45)
    ax.set_xlabel("(read_len, gen_len)")
    ax.grid(alpha=0.3)

axs[0, 0].legend(loc="best")
plt.tight_layout()
combo_path = PLOT_DIR / "profiling_overview.png"
plt.savefig(combo_path)
plt.show()
print(f"Saved {combo_path}")

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

TRACE_DIR = Path("profiling_traces")
TRACE_PLOT_DIR = Path("profiling_plots") / "trace_interpretation"
TRACE_PLOT_DIR.mkdir(parents=True, exist_ok=True)


def load_trace_events(trace_path: Path):
    with open(trace_path, "r") as f:
        obj = json.load(f)
    # Chrome traces from torch.profiler usually store events in traceEvents
    if isinstance(obj, dict) and "traceEvents" in obj:
        return obj["traceEvents"]
    if isinstance(obj, list):
        return obj
    return []


def complete_events(events):
    # ph='X' are complete events with ts/dur
    return [e for e in events if isinstance(e, dict) and e.get("ph") == "X" and "ts" in e and "dur" in e]


def sum_duration_ms(events):
    return sum(float(e.get("dur", 0.0)) for e in events) / 1000.0


def op_duration_map_ms(events):
    acc = {}
    for e in complete_events(events):
        name = e.get("name", "<unknown>")
        acc[name] = acc.get(name, 0.0) + float(e.get("dur", 0.0)) / 1000.0
    return acc


def extract_memory_counters(events):
    # Memory counters may appear as ph='C' entries.
    counters = {}
    for e in events:
        if not isinstance(e, dict) or e.get("ph") != "C":
            continue
        ts = e.get("ts")
        args = e.get("args", {})
        if ts is None or not isinstance(args, dict):
            continue
        for k, v in args.items():
            if isinstance(v, (int, float)):
                counters.setdefault(k, []).append((float(ts) / 1000.0, float(v)))
    for k in counters:
        counters[k].sort(key=lambda x: x[0])
    return counters


def derive_memory_timeline_from_events(events):
    """
    Fallback for traces without ph='C' counters.
    Builds an approximate cumulative memory timeline from memory-like events.
    Returns list[(time_ms, bytes_delta_cumulative)].
    """
    points = []

    def _extract_delta_bytes(args):
        if not isinstance(args, dict):
            return None
        # Common keys seen in profiler traces
        for key in ("Bytes", "bytes", "Alloc size", "Allocation Size", "Requested Size"):
            val = args.get(key)
            if isinstance(val, (int, float)):
                return float(val)
        return None

    for e in events:
        if not isinstance(e, dict):
            continue
        ts = e.get("ts")
        if ts is None:
            continue

        name = str(e.get("name", "")).lower()
        cat = str(e.get("cat", "")).lower()
        args = e.get("args", {})
        delta = _extract_delta_bytes(args)

        # Heuristic: treat memory-tagged events with numeric size info as deltas.
        if delta is None:
            continue
        if "memory" not in name and "memory" not in cat and "alloc" not in name and "alloc" not in cat:
            continue

        # Try to infer sign from event name/cat; unknown defaults to +.
        signed_delta = delta
        if "free" in name or "free" in cat or "dealloc" in name or "dealloc" in cat:
            signed_delta = -abs(delta)

        points.append((float(ts) / 1000.0, signed_delta))

    if not points:
        return []

    points.sort(key=lambda x: x[0])
    running = 0.0
    timeline = []
    for t_ms, d_bytes in points:
        running += d_bytes
        timeline.append((t_ms, running))
    return timeline


# 1) Duration summary across all traces
summary = []
for model_dir in sorted(TRACE_DIR.glob("*")):
    if not model_dir.is_dir():
        continue
    model = model_dir.name

    for prefill_trace in sorted(model_dir.glob("*_prefill.json")):
        base = prefill_trace.stem.replace("_prefill", "")
        decode_trace = model_dir / f"{base}_decode.json"

        if not decode_trace.exists():
            continue

        prefill_events = load_trace_events(prefill_trace)
        decode_events = load_trace_events(decode_trace)

        summary.append(
            {
                "model": model,
                "point": base,  # e.g. r512_g16
                "prefill_ms": sum_duration_ms(complete_events(prefill_events)),
                "decode_ms": sum_duration_ms(complete_events(decode_events)),
                "prefill_trace": str(prefill_trace),
                "decode_trace": str(decode_trace),
            }
        )

if not summary:
    raise FileNotFoundError(f"No paired prefill/decode traces found under {TRACE_DIR}")

# Plot per-model grouped bars for each point (prefill/decode)
for model in sorted({r["model"] for r in summary}):
    rows = [r for r in summary if r["model"] == model]
    rows = sorted(rows, key=lambda r: r["point"])

    points = [r["point"] for r in rows]
    prefill_vals = [r["prefill_ms"] for r in rows]
    decode_vals = [r["decode_ms"] for r in rows]

    x = list(range(len(points)))
    w = 0.4

    plt.figure(figsize=(10, 5))
    plt.bar([i - w / 2 for i in x], prefill_vals, width=w, label="Prefill", alpha=0.9)
    plt.bar([i + w / 2 for i in x], decode_vals, width=w, label="Decode", alpha=0.9)
    plt.xticks(x, points, rotation=45)
    plt.ylabel("Total traced duration (ms)")
    plt.title(f"{model}: Prefill vs Decode Trace Duration")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out = TRACE_PLOT_DIR / f"{model}_prefill_decode_duration.png"
    plt.savefig(out)
    plt.show()
    print(f"Saved {out}")

# 2) Top ops for one selected trace (first row by default)
selected = summary[0]
selected_events = load_trace_events(Path(selected["prefill_trace"]))
op_map = op_duration_map_ms(selected_events)

# Keep top 20 operations by cumulative traced time
top_ops = sorted(op_map.items(), key=lambda kv: kv[1], reverse=True)[:20]
labels = [k for k, _ in top_ops][::-1]
vals = [v for _, v in top_ops][::-1]

plt.figure(figsize=(10, 7))
plt.barh(labels, vals)
plt.xlabel("Cumulative duration (ms)")
plt.title(f"Top Ops (prefill) - {selected['model']} {selected['point']}")
plt.tight_layout()
out = TRACE_PLOT_DIR / "top_ops_prefill_selected.png"
plt.savefig(out)
plt.show()
print(f"Saved {out}")

# 3) Memory timeline: prefer counters, fallback to memory-like events
mem_counters = extract_memory_counters(selected_events)
if mem_counters:
    # Plot up to first 4 counters for readability
    shown = list(mem_counters.keys())[:4]
    plt.figure(figsize=(10, 6))
    for key in shown:
        ts = [p[0] for p in mem_counters[key]]
        ys = [p[1] for p in mem_counters[key]]
        plt.plot(ts, ys, label=key)
    plt.xlabel("Time (ms)")
    plt.ylabel("Counter value")
    plt.title(f"Memory/Counter Timeline - {selected['model']} {selected['point']} (prefill)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    out = TRACE_PLOT_DIR / "memory_counters_selected_prefill.png"
    plt.savefig(out)
    plt.show()
    print(f"Saved {out}")
else:
    derived = derive_memory_timeline_from_events(selected_events)
    if derived:
        ts = [p[0] for p in derived]
        ys_mb = [p[1] / (1024 ** 2) for p in derived]

        plt.figure(figsize=(10, 6))
        plt.plot(ts, ys_mb)
        plt.xlabel("Time (ms)")
        plt.ylabel("Derived cumulative memory delta (MB)")
        plt.title(f"Derived Memory Timeline - {selected['model']} {selected['point']} (prefill)")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        out = TRACE_PLOT_DIR / "memory_timeline_derived_selected_prefill.png"
        plt.savefig(out)
        plt.show()
        print(f"Saved {out}")
    else:
        print("No explicit memory counters or memory-like allocation events found in selected trace.")